# Assignment 2 — Q1. Used Car Data Wrangling
**Course:** CS5530 — Principles of Data Science  
**Student:** Kai Son - 16401784


In [ ]:
# Q1 — Load the dataset

import pandas as pd
from google.colab import drive

drive.mount("/content/drive")

file_path = "/content/drive/MyDrive/Colab Notebooks/CS-5530/Assignment 2 & 3/train.csv"

cars_raw = pd.read_csv(file_path)
cars = cars_raw.copy()

print("Rows and columns:", cars.shape)
display(cars.head())
display(cars.dtypes)

## Q1(a). Identify Missing Values

In [ ]:
# Q1(a) — Check missing values in every original column

missing_before = pd.DataFrame({
    "Missing_Count": cars.isna().sum(),
    "Missing_Percent": (cars.isna().mean() * 100).round(2)
})

display(missing_before)

## Q1(b). Convert values to numbers

I removed the units from Mileage, Engine, and Power. I kept Mileage_Unit because kmpl and km/kg are different. I checked New_Price units and converted prices to lakhs.

In [14]:
# Q1(b) — Keep mileage units before extracting the numbers

cars["Mileage_Unit"] = (
    cars_raw["Mileage"]
    .astype("string")
    .str.extract(r"(kmpl|km/kg)", expand=False)
)

# Extract the numeric part of each measurement.
for column in ["Mileage", "Engine", "Power"]:
    number_text = (
        cars_raw[column]
        .astype("string")
        .str.split()
        .str[0]
    )

    cars[column] = pd.to_numeric(number_text, errors="coerce")

# Show the original and converted values.
conversion_examples = pd.DataFrame({
    "Mileage_Before": cars_raw["Mileage"],
    "Mileage_After": cars["Mileage"],
    "Mileage_Unit": cars["Mileage_Unit"],
    "Engine_Before": cars_raw["Engine"],
    "Engine_After": cars["Engine"],
    "Power_Before": cars_raw["Power"],
    "Power_After": cars["Power"]
})

display(conversion_examples.head())

,Mileage_Before,Mileage_After,Mileage_Unit,Engine_Before,Engine_After,Power_Before,Power_After
0,19.67 kmpl,19.67,kmpl,1582 CC,1582.0,126.2 bhp,126.20
1,13 km/kg,13.00,km/kg,1199 CC,1199.0,88.7 bhp,88.70
2,20.77 kmpl,20.77,kmpl,1248 CC,1248.0,88.76 bhp,88.76
3,15.2 kmpl,15.20,kmpl,1968 CC,1968.0,140.8 bhp,140.80
4,23.08 kmpl,23.08,kmpl,1461 CC,1461.0,63.1 bhp,63.10


In [ ]:
# Q1(b) — Inspect observed price units before conversion
display(cars_raw['New_Price'].str.split().str[-1].value_counts())


In [18]:
# Q1(b) — Convert New_Price to numeric values in lakhs

price_text = cars_raw["New_Price"].astype("string").str.lower()

price_number = pd.to_numeric(
    price_text.str.split().str[0],
    errors="coerce"
)

is_lakh = price_text.str.contains("lakh", na=False)
is_crore = price_text.str.contains("cr", na=False)

# Keep prices with recognized units.
cars["New_Price"] = price_number.where(is_lakh | is_crore)

# Convert crore to lakh.
cars.loc[is_crore, "New_Price"] = price_number[is_crore] * 100

price_examples = pd.DataFrame({
    "Original_New_Price": cars_raw["New_Price"],
    "New_Price_Lakhs": cars["New_Price"]
})

display(price_examples.dropna().head(10))

,Original_New_Price,New_Price_Lakhs
1,8.61 Lakh,8.61
5,21 Lakh,21.00
8,10.65 Lakh,10.65
13,32.01 Lakh,32.01
18,47.87 Lakh,47.87
26,10.57 Lakh,10.57
32,12.33 Lakh,12.33
34,11.12 Lakh,11.12
42,23.64 Lakh,23.64
45,18.64 Lakh,18.64


## Q1(a), Continued. Fill or drop missing values

I used the median for Engine and Power and the mode for Seats. I dropped New_Price because 86.06% was missing. I removed the two rows without mileage information.

In [19]:
# Q1(a) — Inspect missing values after conversion

missing_after_conversion = pd.DataFrame({
    "Missing_Count": cars.isna().sum(),
    "Missing_Percent": (cars.isna().mean() * 100).round(2)
})

display(missing_after_conversion)

# Compare the mean, median, and range of numeric measurements.
display(cars[["Engine", "Power"]].describe().round(2))

# Find the most common seat count.
display(cars["Seats"].value_counts())

,Missing_Count,Missing_Percent
Unnamed: 0,0,0.00
Name,0,0.00
Location,0,0.00
Year,0,0.00
Kilometers_Driven,0,0.00
Fuel_Type,0,0.00
Transmission,0,0.00
Owner_Type,0,0.00
Mileage,2,0.03
Engine,36,0.62


,Engine,Power
count,5811.00,5811.00
mean,1631.55,113.80
std,601.97,53.90
min,72.00,34.20
25%,1198.00,78.00
50%,1497.00,98.60
75%,1991.00,139.01
max,5998.00,560.00


,count
Seats,
5.0,4866
7.0,668
8.0,133
4.0,93
6.0,29
2.0,13
10.0,4
9.0,3


In [ ]:
# Q1(a) — Handle missing values

# Drop the column with extensive missing data.
print("Missing New_Price values:", cars["New_Price"].isna().sum())
cars = cars.drop(columns="New_Price")

# Fill missing Engine and Power values with their medians.
for column in ["Engine", "Power"]:
    missing_count = cars[column].isna().sum()
    median_value = cars[column].median()

    cars[column] = cars[column].fillna(median_value)

    print(
        f"{column}: filled {missing_count} missing values "
        f"with median {median_value:.2f}"
    )

# Fill missing Seats values with the most frequent seat count.
missing_seats = cars["Seats"].isna().sum()
seat_mode = cars["Seats"].mode().iloc[0]

cars["Seats"] = cars["Seats"].fillna(seat_mode)

print(
    f"Seats: filled {missing_seats} missing values "
    f"with mode {seat_mode:g}"
)

# Drop rows whose mileage cannot be interpreted.
rows_before = len(cars)

cars = cars.dropna(subset=["Mileage", "Mileage_Unit"]).copy()

rows_dropped = rows_before - len(cars)

print("Rows dropped because of missing mileage or units:", rows_dropped)
print("Rows remaining:", len(cars))

# Verify all columns after cleaning.
display(cars.isna().sum().to_frame("Missing_After_Cleaning"))


## Q1(c). One-Hot Encode Fuel_Type and Transmission

In [ ]:
# Q1(c) — One-hot encode fuel type and transmission
cars_encoded = pd.get_dummies(
    cars, columns=['Fuel_Type', 'Transmission'], dtype=int
)
display(cars_encoded.head())


## Q1(d). Create Car_Age

I subtracted Year from 2026. I used a fixed year so the result stays the same when I rerun the notebook.

In [ ]:
# Q1(d) — Create Car_Age

reference_year = 2026

cars["Car_Age"] = reference_year - cars["Year"]
cars_encoded["Car_Age"] = reference_year - cars_encoded["Year"]

display(cars[["Name", "Year", "Car_Age"]].head())


## Q1(e). Data manipulation

Select, filter, and rename use a smaller table. Mutate, sort, and group use all cleaned rows. Km_Per_Year uses age in 2026, so it is only an estimate. The odometer date is unknown.

In [24]:
# Q1(e) — Select columns

selected = cars[["Name", "Year", "Fuel_Type", "Price"]]

display(selected.head())

,Name,Year,Fuel_Type,Price
0,Hyundai Creta 1.6 CRDi SX Option,2015,Diesel,12.50
1,Honda Jazz V,2011,Petrol,4.50
2,Maruti Ertiga VDI,2012,Diesel,6.00
3,Audi A4 New 2.0 TDI Multitronic,2013,Diesel,17.74
4,Nissan Micra Diesel XV,2013,Diesel,3.50


In [ ]:
# Q1(e) — Filter cars manufactured in 2011 or earlier
filtered = selected.loc[selected['Year'] <= 2011].copy()
print('Cars manufactured in 2011 or earlier:', len(filtered))
display(filtered.head())


In [28]:
# Q1(e) — Rename Price to show its unit

renamed = filtered.rename(columns={"Price": "Used_Price_Lakhs"})

display(renamed.head())

,Name,Year,Fuel_Type,Used_Price_Lakhs
1,Honda Jazz V,2011,Petrol,4.50
21,Hyundai i20 1.2 Magna,2010,Petrol,1.87
22,Volkswagen Vento Petrol Highline AT,2010,Petrol,2.85
27,Toyota Innova 2.5 V Diesel 7-seater,2007,Diesel,4.00
31,Audi A6 2.7 TDI,2010,Diesel,11.50


In [29]:
# Q1(e) — Mutate: calculate average kilometers driven per year
mutated = cars.copy()

# Replace age 0 with 1 to avoid division by zero
mutated["Km_Per_Year"] = (
    mutated["Kilometers_Driven"] / mutated["Car_Age"].clip(lower=1)
).round(2)

display(mutated[["Name", "Kilometers_Driven", "Car_Age", "Km_Per_Year"]].head())

,Name,Kilometers_Driven,Car_Age,Km_Per_Year
0,Hyundai Creta 1.6 CRDi SX Option,41000,11,3727.27
1,Honda Jazz V,46000,15,3066.67
2,Maruti Ertiga VDI,87000,14,6214.29
3,Audi A4 New 2.0 TDI Multitronic,40670,13,3128.46
4,Nissan Micra Diesel XV,86999,13,6692.23


In [31]:
# Q1(e) — Sort cars by average kilometers per year, highest first
arranged = mutated.sort_values(by="Km_Per_Year", ascending=False)

display(arranged[["Name", "Kilometers_Driven", "Car_Age", "Km_Per_Year"]].head())

,Name,Kilometers_Driven,Car_Age,Km_Per_Year
2259,BMW X5 xDrive 30d M Sport,6500000,9,722222.22
326,Skoda Octavia Ambition Plus 2.0 TDI AT,775000,13,59615.38
1805,Volkswagen Vento Diesel Highline,720000,13,55384.62
2734,Volkswagen Jetta 2013-2015 2.0L TDI Highline AT,480000,11,43636.36
2997,Honda City i VTEC SV,480000,11,43636.36


In [33]:
# Q1(e) — Summarize average kilometers per year by fuel type
summary = mutated.groupby("Fuel_Type", as_index=False).agg(
    Number_of_Cars=("Km_Per_Year", "size"),
    Average_Km_Per_Year=("Km_Per_Year", "mean")
)

summary = summary.sort_values("Average_Km_Per_Year", ascending=False)

display(summary.round(2))

,Fuel_Type,Number_of_Cars,Average_Km_Per_Year
0,Diesel,3161,5495.35
1,Petrol,2684,3580.90


In [ ]:
# Q1(e) — Report the grouped results
for _, row in summary.iterrows():
    print(
        f"{row['Fuel_Type']} has {int(row['Number_of_Cars'])} listings "
        f"with an estimated average of "
        f"{row['Average_Km_Per_Year']:,.2f} kilometers per year."
    )


In [ ]:
# Q1 — Save the final dataset
import os

final_data = cars_encoded.copy()
final_data['Km_Per_Year'] = mutated['Km_Per_Year']

output_folder = '/content/drive/MyDrive/Colab Notebooks/CS-5530/Assignment 2 & 3/Q1_Used_Cars/data_clean'
os.makedirs(output_folder, exist_ok=True)
final_data.to_csv(output_folder + '/cars_final.csv', index=False)
